# NB47 — TabPFN-2.5: Tum Panellerde Ham COMBINED + Reverse-Pool

NB47 -- TabPFN-2.5: Tum Panellerde Ham COMBINED + Reverse-Pool Denemesi

Amac: reports/literature_research_panel_improvements_2026-07-24.md yol haritasi
madde 3. Kurulu tabpfn paketi (v8.0.8) MAX_NUMBER_OF_FEATURES=2000,
MAX_NUMBER_OF_SAMPLES=50_000 destekliyor (TabPFN-2.5 nesli, dogrulandi --
500 feature ustunde estimator-basi subsample yapiyor). NB24'te PAH'a TabPFN
zaten uygulanmisti ama top-50 feature'a KISITLANARAK (o zamanki paket <100
feature tercih ediyordu) ve SADECE ham COMBINED + label-shift ekseninde.
Bu notebook iki farkli eksen acar:
  (a) HAM feature seti (288-353, top-K kisitlamasi YOK) -- artik limitin
      cok altinda.
  (b) TUM 4 panelde (NB24 sadece PAH'ta calisti).
  (c) reverse-pool (%60B/%40P) ile TabPFN'in ilk kez birlikte denenmesi.

Sabit champion recete YOK -- bu kesif amacli bir deney (TabPFN yeni bir
inductive bias). Referans: her panelin mevcut sampiyonu (bkz. progress.md).

Degerlendirme protokolu NB39/NB44/NB46 ile birebir ayni: f1_raw / f1_8020 /
mcc_8020 ayrimi, floor-F1, %80/20 bootstrap (N=50, %95 CI), train-test gap.

Metodolojik not: TabPFN in-context learning oldugu icin ayri bir egitim
dongusu yok -- 'fit' aslinda konteksti hafizaya aliyor, 'predict_proba'
transformer forward-pass yapiyor. CPU'da COMBINED (~3400 satir) icin
inference suresi GBDT'den kat kat uzun olabilir; n_estimators dusuk tutuldu.

In [ ]:
import os, sys, json, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, matthews_corrcoef,
    average_precision_score, confusion_matrix
)

from tabpfn import TabPFNClassifier

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
sys.path.insert(0, '/Users/tefe/teknofest_model/teknofest_model')
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

# --- Sabitler ---
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123
PANEL_SPLIT_FRAC = 0.50
HIGH_MISS_THR = 0.50
TABPFN_N_ESTIMATORS = 8  # NB24 ile ayni (hiz/performans dengesi)

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v29_tabpfn_all_panels')
os.makedirs(RESULTS_DIR, exist_ok=True)
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'SEED={SEED}, PROJECT_ROOT={PROJECT_ROOT}')
print(f'Results -> {RESULTS_DIR}')

ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

In [ ]:
# ============================================================================
# Cell: Veri Yukleme + Sutun Temizligi (NB32/NB39/NB46 ile ayni pattern)
# ============================================================================
data_dir = os.path.join(PROJECT_ROOT, 'data', 'real_data')
df_master = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_MASTER.csv'))
df_kanser = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_KANSER.csv'))
df_cftr = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_CFTR.csv'))
df_pah = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_PAH.csv'))

print(f'MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})')
print(f'KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})')
print(f'CFTR:   {df_cftr.shape} (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})')
print(f'PAH:    {df_pah.shape} (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})')

feat_cols_raw = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

for name, df_panel in [('KANSER', df_kanser), ('CFTR', df_cftr), ('PAH', df_pah)]:
    dup_ids = find_exact_dups(df_panel, df_master, feat_cols_raw, TARGET)
    if dup_ids:
        if name == 'KANSER':
            df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
        elif name == 'CFTR':
            df_cftr = df_cftr[~df_cftr[ID_COL].isin(dup_ids)].reset_index(drop=True)
        else:
            df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
        print(f'{name}: {len(dup_ids)} birebir-ayni satir drop edildi')
    else:
        print(f'{name}: birebir-ayni satir yok')

constant_cols = [c for c in feat_cols_raw if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, feat_cols_raw)
drop_cols = set(constant_cols) | dup_drop
keep_cols = [c for c in feat_cols_raw if c not in drop_cols]
print(f'Constant: {len(constant_cols)}, Dup pairs: {len(dup_pairs)} -> drop {len(dup_drop)}')
print(f'Toplam drop: {len(drop_cols)}, Kalan feature: {len(keep_cols)} (TabPFN-2.5 limiti: 2000 -- rahat sinir)')

for _df in [df_master, df_kanser, df_cftr, df_pah]:
    for c in drop_cols:
        if c in _df.columns:
            _df.drop(columns=[c], inplace=True)

df_combined = pd.concat([df_master, df_kanser, df_cftr, df_pah], ignore_index=True)
print(f'\nCOMBINED (tum 4 panel): {df_combined.shape} '
      f'(pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})')

PANELS = {
    'MASTER': df_master,
    'KANSER': df_kanser,
    'CFTR': df_cftr,
    'PAH': df_pah,
}

def panel_5050_split(df):
    pos = df[df[TARGET]==1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET]==0].sample(frac=1.0, random_state=SEED)
    npos = int(round(len(pos) * PANEL_SPLIT_FRAC))
    nneg = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return tr, te

panel_splits = {}
for pname, pdf in PANELS.items():
    tr, te = panel_5050_split(pdf)
    panel_splits[pname] = {'train': tr, 'test': te}
    print(f'{pname} 50/50 split: train={tr.shape} (pos={tr[TARGET].sum()}), '
          f'test={te.shape} (pos={te[TARGET].sum()}, neg={(te[TARGET]==0).sum()})')
    assert te[TARGET].sum() > 0 and (te[TARGET]==0).sum() > 0, f'{pname} split hatasi!'

In [ ]:
# ============================================================================
# Cell: M3 Preprocessing (NB24/NB32 ile ayni -- median impute + high-miss flag)
# ============================================================================
def fit_preprocessor(train_df, keep_cols, target=TARGET):
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps, "keep_cols": keep_cols
    }

def transform_X(df, prep):
    kc = prep["keep_cols"]
    X = df[kc].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

print("\nM3 Preprocessing hazir.")

In [ ]:
# ============================================================================
# Cell: Pool Builder -- P0_HAM_COMBINED (tum 4 panel, orijinal dagilim) ve
#        P1_REVERSE_6040 (COMBINED %60B/%40P gercek resample, hedef panel
#        HARIC -- leakage onlemek icin panel kendi train/test'ine ayrilmis
#        durumda, COMBINED havuzu digerlerinden + hedef panelin train
#        parcasindan olusuyor)
# ============================================================================
def build_pools_for_target(target_panel_name):
    """Hedef panel disindaki panellerin TAMAMI + hedef panelin train parcasi
    ile COMBINED havuzu kurar (test sizintisi yok)."""
    other_panels = [pd_.copy() for pn, pd_ in PANELS.items() if pn != target_panel_name]
    target_train = panel_splits[target_panel_name]['train'].copy()
    combined_ham = pd.concat(other_panels + [target_train], ignore_index=True)

    comb_neg = combined_ham[combined_ham[TARGET]==0]
    comb_pos = combined_ham[combined_ham[TARGET]==1]
    n_neg = len(comb_neg)
    n_pos_target = max(1, int(round(n_neg * 0.40 / 0.60)))
    pos_sample = comb_pos.sample(n=min(n_pos_target, len(comb_pos)), random_state=SEED)
    reverse_pool = pd.concat([comb_neg, pos_sample]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    return {
        'P0_HAM_COMBINED': combined_ham,
        'P1_REVERSE_6040': reverse_pool,
    }

In [ ]:
# ============================================================================
# Cell: TabPFN Fit + Predict Helper
# ============================================================================
def tabpfn_fit_predict(X_train, y_train, X_test):
    clf = TabPFNClassifier(
        n_estimators=TABPFN_N_ESTIMATORS,
        ignore_pretraining_limits=True,
        device='cpu',
        random_state=SEED,
    )
    t0 = time.time()
    clf.fit(X_train, y_train)
    proba_test = clf.predict_proba(X_test)[:, 1]
    proba_train = clf.predict_proba(X_train)[:, 1]  # threshold secimi icin
    elapsed = time.time() - t0
    return proba_train, proba_test, elapsed

In [ ]:
# ============================================================================
# Cell: Degerlendirme Altyapisi (NB32/NB39/NB44/NB46 ile birebir ayni protokol)
# ============================================================================
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def select_threshold_raw(y, prob):
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        thr_scores[thr] = _f1_pos(y, (prob >= thr).astype(int))
    return float(max(thr_scores, key=thr_scores.get))

def floor_f1(y):
    prev = float(np.mean(y))
    return 2 * prev / (1 + prev)

def eval_full(label, y_test, p_test, y_train, p_train):
    thr_raw = select_threshold_raw(y_train, p_train)
    thr_8020 = select_threshold_8020_robust(y_train, p_train)

    yp_raw = (p_test >= thr_raw).astype(int)
    f1_raw = _f1_pos(y_test, yp_raw)

    yp_8020_thr = (p_test >= thr_8020).astype(int)
    f1_5050 = _f1_pos(y_test, yp_8020_thr)
    mcc_5050 = matthews_corrcoef(y_test, yp_8020_thr)
    prec_5050 = precision_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    rec_5050 = recall_score(y_test, yp_8020_thr, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, yp_8020_thr, labels=[0,1]).ravel()
    auprc = average_precision_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0

    boot = bootstrap_8020(y_test, p_test, thr_8020)
    f1_8020 = boot["mean"]

    rng = np.random.RandomState(BOOT_SEED)
    mccs = []
    for _ in range(N_BOOT):
        yb, pb = _resample_8020(y_test, p_test, rng)
        mccs.append(matthews_corrcoef(yb, (pb >= thr_8020).astype(int)))
    mcc_8020 = float(np.mean(mccs))

    yp_train_8020 = (p_train >= thr_8020).astype(int)
    train_f1 = _f1_pos(y_train, yp_train_8020)
    train_mcc = matthews_corrcoef(y_train, yp_train_8020)

    floor = floor_f1(y_test)

    return {
        "label": label,
        "thr_raw": thr_raw, "thr_8020": thr_8020,
        "f1_raw": f1_raw,
        "f1_5050_8020thr": f1_5050, "mcc_5050_8020thr": mcc_5050,
        "prec_5050": prec_5050, "rec_5050": rec_5050,
        "f1_8020_boot": f1_8020, "f1_8020_std": boot["std"],
        "f1_8020_ci_lo": boot["lo"], "f1_8020_ci_hi": boot["hi"],
        "mcc_8020_boot": mcc_8020,
        "auprc": auprc, "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "floor_f1": floor, "gecti_mi_floor": bool(f1_8020 > floor),
        "train_f1": train_f1, "train_mcc": train_mcc,
        "train_test_gap": train_f1 - f1_8020,
    }

print("\nDegerlendirme altyapisi hazir (f1_raw / f1_8020 / mcc_8020 + floor + train-gap).")

In [ ]:
# ============================================================================
# Cell: Referans Sampiyonlar (progress.md'den, 2026-07-28 itibariyla)
# ============================================================================
REFERENCE_CHAMPIONS = {
    'MASTER': {'name': 'S1_6040_balbag_NB39', 'boot_f1': 0.638},
    'KANSER': {'name': 'P9_REVERSE_6040_catboost_with_fe_NB32', 'boot_f1': 0.730},
    'PAH': {'name': 'P4_COMBINED_BalBag_NB21', 'boot_f1': 0.582},
    'CFTR': {'name': 'S0c_PriorShift_NB20', 'boot_f1': 0.863},
}

In [ ]:
# ============================================================================
# Cell: Ana Dongu -- 4 Panel x 2 Senaryo (P0_HAM_COMBINED, P1_REVERSE_6040)
# ============================================================================
all_results = {}
timings = {}

for panel_name in ['CFTR', 'PAH', 'KANSER', 'MASTER']:  # kucukten buyuge (hizli basarisizlik icin)
    print(f"\n{'='*70}\n[PANEL: {panel_name}]\n{'='*70}")
    test_df = panel_splits[panel_name]['test']
    y_test = test_df[TARGET].values

    pools = build_pools_for_target(panel_name)

    for pool_key, pool_df in pools.items():
        exp_key = f"{panel_name}/{pool_key}/tabpfn"
        pos = pool_df[TARGET].sum(); neg = (pool_df[TARGET]==0).sum()
        print(f"\n  [{exp_key}] n_pool={len(pool_df)} (pos={pos}, neg={neg}, "
              f"benign_frac={neg/(pos+neg):.3f})")
        try:
            prep = fit_preprocessor(pool_df, keep_cols=keep_cols)
            X_pool = transform_X(pool_df, prep)
            X_test = transform_X(test_df, prep)
            y_pool = pool_df[TARGET].values

            p_train, p_test, elapsed = tabpfn_fit_predict(X_pool, y_pool, X_test)
            timings[exp_key] = elapsed

            res = eval_full(exp_key, y_test, p_test, y_pool, p_train)
            res["panel"] = panel_name
            res["pool"] = pool_key
            res["n_pool"] = len(pool_df)
            res["elapsed_sec"] = elapsed
            all_results[exp_key] = res

            ref = REFERENCE_CHAMPIONS[panel_name]
            print(f"    f1_raw={res['f1_raw']:.4f}  f1_8020_boot={res['f1_8020_boot']:.4f} "
                  f"+/-{res['f1_8020_std']:.3f} [CI {res['f1_8020_ci_lo']:.3f}-{res['f1_8020_ci_hi']:.3f}]  "
                  f"mcc_8020={res['mcc_8020_boot']:.4f}  floor={res['floor_f1']:.4f}  "
                  f"gecti_mi={res['gecti_mi_floor']}  train_test_gap={res['train_test_gap']:.4f}  "
                  f"sure={elapsed:.1f}s")
            print(f"    Referans sampiyon ({ref['name']}): {ref['boot_f1']:.4f}  "
                  f"delta={res['f1_8020_boot']-ref['boot_f1']:+.4f}")
        except Exception as e:
            print(f"    HATA: {e}")
            import traceback; traceback.print_exc()
            all_results[exp_key] = None

In [ ]:
# ============================================================================
# Cell: Sonuc Derleme + CSV
# ============================================================================
valid_results = {k: v for k, v in all_results.items() if v is not None}
res_df = pd.DataFrame(valid_results.values())
res_df.to_csv(os.path.join(RESULTS_DIR, 'nb47_tabpfn_all_panels_results.csv'), index=False)
print(f"\nSonuclar kaydedildi: {RESULTS_DIR}/nb47_tabpfn_all_panels_results.csv")
print(res_df[["panel", "pool", "f1_raw", "f1_8020_boot", "f1_8020_ci_lo",
              "f1_8020_ci_hi", "mcc_8020_boot", "floor_f1", "gecti_mi_floor",
              "train_test_gap", "elapsed_sec"]].to_string(index=False))

In [ ]:
# ============================================================================
# Cell: Panel-Bazli Karar (TabPFN en iyisi vs referans sampiyon)
# ============================================================================
panel_verdicts = {}
for panel_name in PANELS.keys():
    panel_rows = {k: v for k, v in valid_results.items() if v["panel"] == panel_name}
    if not panel_rows:
        continue
    best_key = max(panel_rows, key=lambda k: panel_rows[k]["f1_8020_boot"])
    best_val = panel_rows[best_key]
    ref = REFERENCE_CHAMPIONS[panel_name]
    delta = best_val["f1_8020_boot"] - ref["boot_f1"]
    panel_verdicts[panel_name] = {
        "best_tabpfn_scenario": best_key,
        "best_tabpfn_boot_f1": best_val["f1_8020_boot"],
        "reference_champion": ref["name"],
        "reference_boot_f1": ref["boot_f1"],
        "delta": delta,
        "verdict": (
            "TabPFN yeni sampiyon (kayda deger kazanc)" if delta > 0.01 else
            "TabPFN mevcut sampiyonu gecemedi" if delta < -0.01 else
            "TabPFN farksiz (gurultu bandinda)"
        )
    }
    print(f"\n{panel_name}: en iyi TabPFN senaryosu={best_key} "
          f"(Boot-F1={best_val['f1_8020_boot']:.4f}), referans={ref['name']} "
          f"(Boot-F1={ref['boot_f1']:.4f}), delta={delta:+.4f} -> {panel_verdicts[panel_name]['verdict']}")

In [ ]:
# ============================================================================
# Cell: Gorsellestirme
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, panel_name in zip(axes.flat, PANELS.keys()):
    panel_rows = {k: v for k, v in valid_results.items() if v["panel"] == panel_name}
    labels = list(panel_rows.keys())
    means = [panel_rows[l]["f1_8020_boot"] for l in labels]
    los = [panel_rows[l]["f1_8020_ci_lo"] for l in labels]
    his = [panel_rows[l]["f1_8020_ci_hi"] for l in labels]
    errs = [[m - lo for m, lo in zip(means, los)], [hi - m for m, hi in zip(means, his)]]
    short_labels = [l.split('/')[1] for l in labels]
    ax.bar(short_labels, means, yerr=errs, capsize=6, color=['#4C72B0', '#DD8452'])
    ref = REFERENCE_CHAMPIONS[panel_name]
    ax.axhline(ref["boot_f1"], color='green', linestyle=':', label=f'Referans ({ref["boot_f1"]:.3f})')
    if panel_rows:
        floor_val = list(panel_rows.values())[0]["floor_f1"]
        ax.axhline(floor_val, color='red', linestyle='--', label='Floor-F1')
    ax.set_title(panel_name)
    ax.set_ylabel('Boot-F1 (%80/20)')
    ax.legend(fontsize=7)
plt.suptitle('NB47 -- TabPFN-2.5 Tum Panellerde (Ham COMBINED vs Reverse-Pool)')
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'nb47_tabpfn_all_panels_comparison.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"\nGorsel kaydedildi: {fig_path}")

In [ ]:
# ============================================================================
# Cell: Ozet JSON
# ============================================================================
summary = {
    "notebook": "NB47",
    "date": "2026-07-28",
    "hypothesis": "TabPFN-2.5 (2000 feature / 50k ornek limiti) 4 panelde ham-COMBINED + reverse-pool ile",
    "tabpfn_n_estimators": TABPFN_N_ESTIMATORS,
    "n_features_used": len(keep_cols),
    "reference_champions": REFERENCE_CHAMPIONS,
    "results": {k: {kk: vv for kk, vv in v.items() if kk not in ("label",)} for k, v in valid_results.items()},
    "panel_verdicts": panel_verdicts,
    "timings_sec": timings,
}
with open(os.path.join(RESULTS_DIR, 'nb47_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\nOzet JSON kaydedildi: {RESULTS_DIR}/nb47_summary.json")

print("\n" + "="*70)
print("NIHAI OZET")
print("="*70)
for panel_name, v in panel_verdicts.items():
    print(f"{panel_name}: {v['verdict']} (delta={v['delta']:+.4f})")